# Exemplar benchmark (reader-facing copy)

This notebook keeps the harder exemplar experiments in one place.

It compares a few exemplar methods on an easy and a harder toy regime using the current matching + sequence-bag pipeline.

**TBD:** Add the neural exemplar extraction.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "path_matcher").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing pyproject.toml and src/path_matcher.")


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
for path in (str(REPO_ROOT), str(SRC)):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Repo root:", REPO_ROOT)


In [ ]:
from pathlib import Path
import sys
import time
import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "exemplar").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing pyproject.toml and src/exemplar.")


ROOT = find_repo_root()
SRC = ROOT / "src"
for path in (str(ROOT), str(SRC)):
    if path not in sys.path:
        sys.path.insert(0, path)

from benchmarking.shared import (
    sample_or_load_dataset,
    compute_or_load_exact_pair_scores,
    compute_or_load_sequence_bags,
)
from exemplar import infer_cluster_exemplars
from exemplar.metrics import evaluate_exemplar, bag_truth_diagnostics

pd.set_option('display.max_colwidth', 160)
pd.set_option('display.max_rows', 200)

In [ ]:
CACHE_ROOT = ROOT / '.cache/benchmarks'

DATASET_SPECS = {
    'toy_small': {
        'name': 'toy_small',
        'toy_model': {
            'n_classes': 3,
            'n_per_class': 8,
            'planted_seq_len': 10,
            'alphabet_size': 8,
            'p_obs': 0.85,
            'seed': 123,
        },
        'tree_sampler': {
            'max_depth': 10,
            'lam': 1.7,
            'max_nodes': 120,
            'min_nodes': 120,
            'traversal': 'bfs',
        },
    },
    'toy_exemplar_hard': {
        'name': 'toy_exemplar_hard',
        'toy_model': {
            'n_classes': 4,
            'n_per_class': 24,
            'planted_seq_len': 22,
            'alphabet_size': 12,
            'p_obs': 0.58,
            'seed': 31415,
        },
        'tree_sampler': {
            'max_depth': 14,
            'lam': 4.0,
            'max_nodes': 260,
            'min_nodes': 220,
            'traversal': 'bfs',
            'require_full_depth': False,
        },
    },
}

METHOD_SPECS = [
    {'name': 'medoid', 'method': 'medoid'},
    {'name': 'poa_auto', 'method': 'poa', 'backend': 'auto', 'seed_mode': 'medoid', 'order_mode': 'closest_first'},
    {
        'name': 'likelihood_simple',
        'method': 'likelihood',
        'engine': 'simple',
        'init_mode': 'medoid',
        'target_total_sequences': 48,
        'max_total_sequences': 96,
        'lower_slack': 4,
        'upper_slack': 10,
        'n_steps': 12000,
        'n_restarts': 4,
        'temperature': 0.82,
    },
    {
        'name': 'likelihood_workflow_seeded',
        'method': 'likelihood',
        'engine': 'workflow_seeded',
        'init_mode': 'medoid',
        'target_total_sequences': 48,
        'max_total_sequences': 96,
        'lower_slack': 4,
        'upper_slack': 10,
        'n_restarts': 2,
        'em_iters': 3,
        'do_mcmc_refine': False,
        'search_subset_size': 64,
    },
]

In [ ]:
def load_bags_for_dataset(dataset_spec, *, cache_root=CACHE_ROOT, matcher_kind='fast', matcher_kwargs=None):
    matcher_kwargs = {} if matcher_kwargs is None else dict(matcher_kwargs)
    bundle = sample_or_load_dataset(dataset_spec, cache_dir=cache_root / 'sampled_datasets', verbose=True)
    pair_payload = compute_or_load_exact_pair_scores(
        bundle.graphs,
        dataset_name=bundle.name,
        dataset_hash=str(bundle.metadata.get('dataset_hash', 'unknown')),
        matcher_kind=matcher_kind,
        matcher_kwargs=matcher_kwargs,
        cache_dir=cache_root / 'exact_similarity_cache',
        verbose=True,
    )
    bag_payload = compute_or_load_sequence_bags(
        bundle.graphs,
        labels=bundle.labels,
        matches=pair_payload['matches'],
        dataset_name=bundle.name,
        dataset_hash=str(bundle.metadata.get('dataset_hash', 'unknown')),
        matcher_kind=matcher_kind,
        matcher_kwargs=matcher_kwargs,
        seq_mode='raw',
        deduplicate=True,
        rebalance_by_tree=True,
        cache_dir=cache_root / 'extracted_sequence_bags',
        verbose=True,
    )
    return bundle, bag_payload['bags']


def summarize_bag_hardness(bundle, bags):
    rows = []
    for cluster, bag in sorted(bags.items()):
        truth = list(bundle.class_sequences[int(cluster)])
        row = {'cluster': int(cluster), 'truth': truth, 'truth_len': len(truth)}
        row.update(bag_truth_diagnostics(bag, truth))
        rows.append(row)
    return pd.DataFrame(rows).sort_values('cluster').reset_index(drop=True)


def benchmark_methods(bundle, bags, method_specs):
    rows = []
    truth_by_cluster = {int(c): list(seq) for c, seq in enumerate(bundle.class_sequences)}
    for spec in method_specs:
        spec = dict(spec)
        label = spec.pop('name')
        method = spec.pop('method')
        t0 = time.perf_counter()
        results = infer_cluster_exemplars(bags, method=method, return_details=True, **spec)
        dt = time.perf_counter() - t0
        for cluster, res in sorted(results.items()):
            exemplar = list(getattr(res, 'exemplar', res))
            row = {
                'dataset': bundle.name,
                'method': label,
                'cluster': int(cluster),
                'runtime_sec_total': dt,
                'backend': getattr(res, 'backend', method),
                'exemplar': exemplar,
            }
            row.update(evaluate_exemplar(
                exemplar,
                truth=truth_by_cluster[int(cluster)],
                bag=bags[int(cluster)],
                other_bags={k: v for k, v in bags.items() if int(k) != int(cluster)},
            ))
            rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
datasets = {}
hardness_tables = {}
for name, spec in DATASET_SPECS.items():
    bundle, bags = load_bags_for_dataset(spec)
    datasets[name] = (bundle, bags)
    hardness_tables[name] = summarize_bag_hardness(bundle, bags)

for name, df in hardness_tables.items():
    print(f"=== {name}: bag hardness ===")
    display(df[['cluster', 'truth_len', 'n_unique_obs', 'n_obs', 'duplicate_fraction', 'truth_present_in_bag', 'best_observed_norm_edit', 'bag_min_length', 'bag_median_length', 'bag_max_length']])

In [ ]:
all_rows = []
for name, (bundle, bags) in datasets.items():
    df = benchmark_methods(bundle, bags, METHOD_SPECS)
    all_rows.append(df)

results = pd.concat(all_rows, ignore_index=True)
display(results.sort_values(['dataset', 'method', 'cluster']))

In [ ]:
summary = (
    results.groupby(['dataset', 'method'], as_index=False)
    .agg(
        mean_norm_edit_to_truth=('norm_edit_to_truth', 'mean'),
        median_norm_edit_to_truth=('norm_edit_to_truth', 'median'),
        mean_within_mean_norm_edit=('within_mean_norm_edit', 'mean'),
        mean_separation_gap=('separation_gap', 'mean'),
        mean_length_ratio_to_truth=('length_ratio_to_truth', 'mean'),
        runtime_sec_total=('runtime_sec_total', 'mean'),
    )
    .sort_values(['dataset', 'mean_norm_edit_to_truth', 'mean_within_mean_norm_edit'])
    .reset_index(drop=True)
)
summary

In [ ]:
for name in DATASET_SPECS:
    print(f"
=== {name}: per-cluster details ===")
    display(
        results.loc[results['dataset'] == name, [
            'method', 'cluster', 'exemplar', 'norm_edit_to_truth',
            'within_mean_norm_edit', 'separation_gap', 'length_ratio_to_truth',
        ]].sort_values(['method', 'cluster']).reset_index(drop=True)
    )